# SignalForge — train and evaluate

Reproduce the app's offline feature, chronological model-evaluation and costed-backtest pipeline. The bundled sample is synthetic and for software demonstration only; replace it with a licensed, cleaned NSE OHLCV dataset before interpreting results.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'backend'))

import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline

from app.data.market_data import MarketDataStore
from app.features.pipeline import FEATURE_COLUMNS, model_frame
from app.backtest.engine import run_backtest

store = MarketDataStore('backend/data/demo_ohlcv.csv')
bars = store.get('RELIANCE')
print(f'Loaded {len(bars):,} bars across {bars.datetime.dt.date.nunique()} sessions')

## Feature engineering and leakage-aware labels

Features are trailing calculations. The forward label uses the next three closes and a volatility band known at the decision point. Labels that cross a session boundary and the final horizon rows are dropped. The train/validation/test blocks remain chronological.

In [ ]:
dataset = model_frame(bars, horizon=3)
x, y = dataset[FEATURE_COLUMNS], dataset['label'].astype(int)
train_end, validation_end = int(len(dataset) * .60), int(len(dataset) * .80)
# Purge the final three labels before each split so their forward horizon cannot touch the next block.
x_train, y_train = x.iloc[:train_end - 3], y.iloc[:train_end - 3]
x_validation, y_validation = x.iloc[train_end:validation_end - 3], y.iloc[train_end:validation_end - 3]
x_test, y_test = x.iloc[validation_end:], y.iloc[validation_end:]
print({'train': len(x_train), 'validation': len(x_validation), 'test': len(x_test)})
print('Label distribution (train):')
display(y_train.value_counts(normalize=True).sort_index().rename('share').to_frame())

In [ ]:
model = make_pipeline(
    SimpleImputer(),
    HistGradientBoostingClassifier(max_iter=100, max_leaf_nodes=15, l2_regularization=1.0, random_state=42),
)
model.fit(x_train, y_train)
for split_name, split_x, split_y in [('train', x_train, y_train), ('validation', x_validation, y_validation), ('test', x_test, y_test)]:
    predictions = model.predict(split_x)
    print(f'{split_name:>10} | accuracy={accuracy_score(split_y, predictions):.3f} | macro-F1={f1_score(split_y, predictions, average="macro", zero_division=0):.3f}')

test_predictions = model.predict(x_test)
print('\nFinal chronological test classification report:')
print(classification_report(y_test, test_predictions, labels=[-1, 0, 1], target_names=['SELL', 'HOLD', 'BUY'], zero_division=0))
display(pd.DataFrame(confusion_matrix(y_test, test_predictions, labels=[-1, 0, 1]), index=['actual SELL', 'actual HOLD', 'actual BUY'], columns=['pred SELL', 'pred HOLD', 'pred BUY']))

## Cost-aware intraday backtest

The included engine uses a lagged-momentum baseline for reproducible offline simulations, ATR-derived protective levels, a three-bar holding limit, end-of-session closure and configurable basis-point costs. It is intentionally not presented as a deployable strategy.

In [ ]:
backtest = run_backtest(bars, threshold=.55, stop_atr=1.0, target_atr=1.5, capital=100_000, cost_bps=8, horizon=3)
print('Demo backtest metrics:')
display(pd.Series(backtest['metrics'], name='value').to_frame())
trades = pd.DataFrame(backtest['trades'])
display(trades.head(10))
if backtest['equity_curve']:
    equity = pd.DataFrame(backtest['equity_curve'])
    equity['datetime'] = pd.to_datetime(equity['datetime'])
    equity.set_index('datetime')['equity'].plot(title='Illustrative equity curve', figsize=(12, 4), grid=True)